# Whisper large-v3-turbo — DIMER ASR tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/whisper-asr-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/whisper-asr-pipeline/blob/main/tutorials/whisper_asr_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-openai%2Fwhisper--large--v3--turbo-ffcc4d?style=flat)](https://huggingface.co/openai/whisper-large-v3-turbo) [![Upstream](https://img.shields.io/badge/Upstream-openai%2Fwhisper-181717?style=flat&logo=github&logoColor=white)](https://github.com/openai/whisper) [![arXiv](https://img.shields.io/badge/arXiv-2212.04356-b31b1b.svg)](https://arxiv.org/abs/2212.04356)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.1 — **standalone** (§3.6)  
**Capability:** multilingual automatic speech recognition (transcribe or translate-to-English) using the pinned `openai/whisper-large-v3-turbo` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/whisper_asr_pipeline/pipeline.py` at revision `c0b9e1100b0e`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `41f01f3fe87f28c78e2fbf8b568835947dd65ed9` (~1622 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

At inference the encoder maps 30-second windows of 128-bin log-Mel features to hidden states and the 4-layer decoder generates text tokens autoregressively; the transformers ASR pipeline handles resampling, feature extraction and chunking of longer audio, and the carried pipeline module returns the normalised transcript plus provenance. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights, tokenizer and feature-extractor configuration, and the carried module adds snapshot verification, the input contract, a fixed output contract and the `word_error_rate`, `validate_inputs` and `evaluation_report` helpers. The default sample is one public LibriSpeech utterance with its reference transcript, fetched at a pinned dataset revision; its WER is demonstration (plumbing) evidence for one utterance, not a benchmark claim.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, load one public referenced utterance (or upload your own audio) and validate it into an input manifest, run the supported task, read the generated transcript correctly, exercise an optional BYOD path, produce an evaluation report that is `sample-sanity` with `word_error_rate` only when a reference transcript exists and `not-measurable` otherwise, and export machine-readable outputs plus provenance.

**This notebook does not demonstrate:** speaker diarization, speaker identification or any biometric inference, word-level confidence or a transcript-acceptance threshold, streaming/real-time recognition, text-to-speech, or any training. Whisper can hallucinate fluent text on silence, music or non-speech audio, and the pipeline does not detect it.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (float16); the pinned `torch==2.6.0` install and the 1.6 GB checkpoint fetch are the largest downloads of the run.
- **Knowledge:** basic Python; what a waveform, a sampling rate and a word error rate are.
- **Data:** the default sample is the first validation utterance of the public `hf-internal-testing/librispeech_asr_dummy` dataset (a few seconds of read English speech with its reference transcript), fetched from the Hugging Face Hub at a pinned dataset revision and decoded with the pinned `soundfile` dependency — no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one audio file readable by the ASR stack (WAV/FLAC/MP3 and similar, mono or stereo); if you know its transcript, paste it into `REFERENCE_TEXT` so the evaluation step can compute `word_error_rate`. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `openai/whisper-large-v3-turbo` snapshot (~1622 MB in total) at revision `41f01f3fe87f…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.6.0',
    'torchvision==0.21.0',
    'torchaudio==2.6.0',
    'transformers==4.52.1',
    'accelerate==1.3.0',
    'huggingface-hub==0.36.2',
    'numpy==1.26.4',
    'soundfile==0.13.1',
    'datasets==4.4.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'whisper-asr-pipeline',
    'repository_revision': 'c0b9e1100b0ebe28639a0c98c31d967a18f7aba3',
    'embedded_module': 'src/whisper_asr_pipeline/pipeline.py',
    'embedded_modules': ['src/whisper_asr_pipeline/pipeline.py'],
    'module_sha256': '93245d3dbdca6afc5b1cc949330090a9f8763915f5ce0ea4dcf2e8a2d29bf2d4',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '1.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/whisper_asr_pipeline/` @ `c0b9e1100b0e`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/whisper_asr_pipeline/pipeline.py`

In [ ]:
"""Multilingual speech recognition with the pinned ``openai/whisper-large-v3-turbo`` checkpoint.

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the architecture comes from the pinned ``transformers`` release, the
weights are SafeTensors, and no model-repository code is executed.
"""

from __future__ import annotations

import hashlib
import json
import re
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

MODEL_ID = "openai/whisper-large-v3-turbo"
MODEL_REVISION = "41f01f3fe87f28c78e2fbf8b568835947dd65ed9"
MODEL_LICENSE = "MIT"
MODEL_KEY = "whisper-large-v3-turbo"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHTS_FILE = "model.safetensors"
CONFIG_FILE = "config.json"

TASKS = ("transcribe", "translate")
MIN_CHUNK_LENGTH_S = 1
MAX_CHUNK_LENGTH_S = 30  # Whisper's receptive field; longer audio is chunked by the transformers pipeline
# Basic WER normalization: case-fold and drop punctuation so that "classes," and "gospel."
# match an unpunctuated reference. Curly apostrophes are folded to the straight form first;
# word-internal apostrophes and hyphens are kept (a hyphenated compound stays one token).
# Numbers, abbreviations and spelled-out forms are NOT normalized ("Mr." vs "Mister" is an
# error).
_APOSTROPHES = str.maketrans({"\u2019": "'", "\u2018": "'", "\u02bc": "'"})
_PUNCTUATION = re.compile(r"[^\w\s'-]|(?<!\w)['-]|['-](?!\w)", re.UNICODE)


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path or DEFAULT_WEIGHTS_DIR)
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest and the
    tokenizer/config files but git-ignores the weights). Returns the relative paths fetched;
    `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _tokens(text: str) -> list[str]:
    return _PUNCTUATION.sub(" ", text.casefold().translate(_APOSTROPHES)).split()


def word_error_rate(reference: str, hypothesis: str) -> float:
    """Word error rate after basic normalization (lowercase, punctuation removed)."""
    reference_tokens = _tokens(reference)
    hypothesis_tokens = _tokens(hypothesis)
    if not reference_tokens:
        return 0.0 if not hypothesis_tokens else 1.0

    previous = list(range(len(hypothesis_tokens) + 1))
    for row_index, reference_token in enumerate(reference_tokens, 1):
        current = [row_index]
        for column_index, hypothesis_token in enumerate(hypothesis_tokens, 1):
            current.append(
                min(
                    current[-1] + 1,
                    previous[column_index] + 1,
                    previous[column_index - 1] + (reference_token != hypothesis_token),
                )
            )
        previous = current
    return previous[-1] / len(reference_tokens)


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one audio input: a local file path readable by the ASR stack, an http(s) URL, or a dict "
        "{'array': float waveform, 'sampling_rate': int}"
    ),
    "task": list(TASKS),
    "language": "optional ISO language name/code forwarded to Whisper; None lets the model detect it",
    "chunk_length_s": [MIN_CHUNK_LENGTH_S, MAX_CHUNK_LENGTH_S],
    "preprocessing": (
        "the transformers ASR pipeline resamples to 16 kHz, computes 128-bin log-Mel features in 30 s "
        "windows and chunks longer audio; nothing is altered by this module"
    ),
}


def _check_inputs(audio: Any, task: str, chunk_length_s: int) -> None:
    """Raise ValueError/FileNotFoundError naming the first violated rule (shared with transcribe)."""
    if task not in TASKS:
        raise ValueError("task must be 'transcribe' or 'translate'")
    if (
        isinstance(audio, str | Path)
        and not str(audio).startswith(("http://", "https://"))
        and not Path(audio).is_file()
    ):
        raise FileNotFoundError(f"audio file not found: {audio}")
    if not MIN_CHUNK_LENGTH_S <= chunk_length_s <= MAX_CHUNK_LENGTH_S:
        raise ValueError(
            f"chunk_length_s must be between {MIN_CHUNK_LENGTH_S} and {MAX_CHUNK_LENGTH_S} seconds"
        )


def _observe(audio: Any) -> dict[str, Any]:
    if isinstance(audio, Mapping):
        array = audio.get("array")
        rate = audio.get("sampling_rate")
        samples = len(array) if array is not None and hasattr(array, "__len__") else None
        has_rate = isinstance(rate, int | float) and bool(rate)
        seconds = round(samples / rate, 3) if samples is not None and has_rate else None
        return {"kind": "waveform", "samples": samples, "sampling_rate": rate, "seconds": seconds}
    if isinstance(audio, str | Path) and str(audio).startswith(("http://", "https://")):
        return {"kind": "url", "value": str(audio)}
    path = Path(audio)
    return {"kind": "file", "name": path.name, "bytes": path.stat().st_size}


def validate_inputs(
    audio: str | Path | Mapping[str, Any],
    *,
    language: str | None = None,
    task: str = "transcribe",
    chunk_length_s: int = 30,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observed input, request, verdict).

    Rejection is reported by raising exactly as ``transcribe`` would; a caller that wants the
    finding recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _check_inputs(audio, task, chunk_length_s)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (one audio input per call)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "audio-0", **_observe(audio)}],
        "task": task,
        "language": language,
        "chunk_length_s": chunk_length_s,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], reference: str | None = None, *, sample_kind: str = "public-sample"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With a ``reference`` transcript the report carries ``word_error_rate`` (the repository's own
    normalised WER) as sample-sanity evidence for that one utterance; without one the verdict is
    ``not-measurable`` and the report says what would make the task measurable.
    """
    base = {
        "task": f"automatic speech recognition ({result.get('task', 'transcribe')})",
        "score_semantics": "generated transcript; the pipeline exposes no confidence score or threshold",
        "sample_kind": sample_kind,
        "n_utterances": 1,
        "language": result.get("language"),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if reference is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no reference transcript was supplied for the evaluated audio",
            "needs": (
                "reference transcripts for audio from the deployment domain (speakers, microphones, noise), "
                "scored with word_error_rate after the same normalisation; several hundred utterances before "
                "any rate is quoted"
            ),
        }
    return {
        **base,
        "metrics": [
            {
                "id": "word_error_rate",
                "value": word_error_rate(reference, str(result["text"])),
                "normalisation": "casefold, punctuation removed, curly apostrophes folded",
                "estimation": "single utterance, no dispersion estimate",
            }
        ],
        "verdict": "sample-sanity",
        "reason": "one referenced utterance from the tutorial sample; not a benchmark",
        "needs": "a referenced evaluation set from the deployment domain for any generalisable WER claim",
    }


@dataclass
class WhisperASRPipeline:
    _runner: Callable[..., dict[str, Any]]
    device: str
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> WhisperASRPipeline:
        import torch
        from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        dtype = torch.float16 if resolved_device.startswith("cuda") else torch.float32
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            # A directory argument makes transformers read config/tokenizer/weights from it directly
            # (no Hub resolution, no cache lookup).
            location: dict[str, Any] = {"pretrained_model_name_or_path": str(root)}
            source = "local-snapshot"
        elif allow_download:
            location = {"pretrained_model_name_or_path": MODEL_ID, "revision": MODEL_REVISION}
            source = "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        processor = AutoProcessor.from_pretrained(**location, trust_remote_code=False)
        model = AutoModelForSpeechSeq2Seq.from_pretrained(
            **location,
            torch_dtype=dtype,
            low_cpu_mem_usage=True,
            trust_remote_code=False,
        )
        if resolved_device.startswith("cuda"):
            model = model.to(resolved_device)
        runner = pipeline(
            "automatic-speech-recognition",
            model=model,
            tokenizer=processor.tokenizer,
            feature_extractor=processor.feature_extractor,
            torch_dtype=dtype,
            device=resolved_device,
        )
        return cls(runner, resolved_device, source)

    def transcribe(
        self,
        audio: str | Path | dict[str, Any],
        *,
        language: str | None = None,
        task: str = "transcribe",
        return_timestamps: bool = False,
        chunk_length_s: int = 30,
    ) -> dict[str, Any]:
        _check_inputs(audio, task, chunk_length_s)

        generate_kwargs: dict[str, Any] = {"task": task}
        if language:
            generate_kwargs["language"] = language
        raw = self._runner(
            str(audio) if isinstance(audio, Path) else audio,
            return_timestamps=return_timestamps,
            chunk_length_s=chunk_length_s,
            generate_kwargs=generate_kwargs,
        )
        if not isinstance(raw, dict) or "text" not in raw:
            raise RuntimeError("ASR backend returned an invalid result")
        return {
            "text": str(raw["text"]).strip(),
            "chunks": raw.get("chunks"),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "task": task,
            "language": language,
            "device": self.device,
            "source": self.source,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `12`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `41f01f3fe87f…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `WhisperASRPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "whisper-large-v3-turbo",
  "modelId": "openai/whisper-large-v3-turbo",
  "revision": "41f01f3fe87f28c78e2fbf8b568835947dd65ed9",
  "files": [
    {
      "path": "README.md",
      "bytes": 21196,
      "sha256": "aaef74a740faca90fa1899c4233ebe17f2093b9846d0acf16d9131bf650e9585"
    },
    {
      "path": "added_tokens.json",
      "bytes": 34648,
      "sha256": "3c51f66c4c21f9e126970078f11ae77a78c74aee8df606ee9daba86e467108e0"
    },
    {
      "path": "config.json",
      "bytes": 1256,
      "sha256": "c5b526b3e3cd64cd8940dabb45e8ba726629e22d8ed389c29b552f9140daf04a"
    },
    {
      "path": "generation_config.json",
      "bytes": 3772,
      "sha256": "cce11bfe3aaa6ae9e072ea2637caaec8795e68d9b67e655a5af16ee509681a4c"
    },
    {
      "path": "merges.txt",
      "bytes": 493869,
      "sha256": "2df2990a395e35e8dfbc7511e08c12d56018d8d04691e0133e5d63b21e154dc6"
    },
    {
      "path": "model.safetensors",
      "bytes": 1617824864,
      "sha256": "542566a422ae4f3fd23f1ba11add198fca01bbf82e66e6a2857b3f608b1eb9d1"
    },
    {
      "path": "normalizer.json",
      "bytes": 52666,
      "sha256": "bf1c507dc8724ca9cf9903640dacfb69dae2f00edee4f21ceba106a7392f26dd"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 340,
      "sha256": "7ccc62c6f2765af1f3b46c00c9b5894426835a05021c8b9c01eecb6dfb542711"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 2186,
      "sha256": "baea4ea09372eb4fca86b4e4346139fd73cb807d5087e9de0948e971739c3e74"
    },
    {
      "path": "tokenizer.json",
      "bytes": 2710337,
      "sha256": "297b13372ac43916285644fb9687add3cc62ee2a1adb60da3dc25cc94c1871fd"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 282843,
      "sha256": "844b642c73a91359722f47b35705f7174686df33d252695d8572cf9ac03a6389"
    },
    {
      "path": "vocab.json",
      "bytes": 1036558,
      "sha256": "e2aa043ef015641d363d8288e7c241c85e36a5c761fb303598e0710233344387"
    }
  ],
  "totalBytes": 1622464535
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = WhisperASRPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Load the public sample or optional BYOD

The default sample is **public**: the first `validation` row of `hf-internal-testing/librispeech_asr_dummy` (`clean` config), loaded at the pinned dataset revision `5be91486e11a2d616f4ec5db8d3fd248585ac07a` so the bytes cannot drift; the stored audio bytes are decoded with the pinned `soundfile` dependency into a float32 waveform plus sampling rate (the datasets audio feature would decode through torchcodec/FFmpeg, which this runtime does not pin), stereo is averaged to mono, and the waveform's SHA-256 is printed for the record. Its reference transcript gives the evaluation step a ground truth. BYOD is optional and disabled by default; when enabled, upload one audio file and, if you know its transcript, set `REFERENCE_TEXT` (leave it empty when unknown). Look for a dictionary naming the sample kind, its duration and digest, and whether a reference exists.

In [ ]:
import hashlib
import io

import numpy as np
import soundfile as sf
from datasets import Audio, load_dataset

USE_BYOD = False  # @param {type:"boolean"}
REFERENCE_TEXT = ''  # @param {type:"string"}
SAMPLE_DATASET = 'hf-internal-testing/librispeech_asr_dummy'
SAMPLE_DATASET_REVISION = '5be91486e11a2d616f4ec5db8d3fd248585ac07a'

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next(iter(uploaded))
    audio_input = sample_name
    waveform, sampling_rate = sf.read(io.BytesIO(uploaded[sample_name]), dtype='float32')
    if waveform.ndim > 1:
        waveform = waveform.mean(axis=1)
    reference = REFERENCE_TEXT.strip() or None
    sample_kind = 'BYOD'
else:
    ds = load_dataset(SAMPLE_DATASET, 'clean', split='validation', revision=SAMPLE_DATASET_REVISION)
    ds = ds.cast_column('audio', Audio(decode=False))
    row = ds[0]
    waveform, sampling_rate = sf.read(io.BytesIO(row['audio']['bytes']), dtype='float32')
    if waveform.ndim > 1:
        waveform = waveform.mean(axis=1)
    audio_input = {'array': waveform, 'sampling_rate': sampling_rate}
    sample_name = f"{SAMPLE_DATASET}:clean:validation[0] ({row['id']})"
    reference = row['text']
    sample_kind = 'public-sample'

sample_sha256 = hashlib.sha256(np.ascontiguousarray(waveform, dtype=np.float32).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': sample_name, 'seconds': round(len(waveform) / sampling_rate, 2), 'sampling_rate': sampling_rate, 'waveform_sha256': sample_sha256, 'has_reference': reference is not None})

## 5. Validate the input → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `transcribe` applies — the task must be `transcribe` or `translate`, a path must exist, `chunk_length_s` must be 1..`MAX_CHUNK_LENGTH_S` — and returns an **input manifest** naming the schema and ceilings, the observed input (kind, samples, sampling rate, duration), the request (task, language, chunk length) and the verdict. The manifest is written to `outputs/whisper_asr_input_manifest.json`. To show what rejection looks like, the cell also validates a deliberately oversized chunk length and records the pipeline's own error message as a finding. Inside the transformers pipeline the audio is resampled to 16 kHz and turned into 30-second log-Mel windows; nothing else is dropped or altered.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'TASKS': list(TASKS), 'MIN_CHUNK_LENGTH_S': MIN_CHUNK_LENGTH_S, 'MAX_CHUNK_LENGTH_S': MAX_CHUNK_LENGTH_S}})
input_manifest = validate_inputs(audio_input, language='en', task='transcribe', names=[sample_name])
# Demonstrate rejection on a request that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(audio_input, chunk_length_s=MAX_CHUNK_LENGTH_S + 1)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'oversized-chunk-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/whisper_asr_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Transcribe

`transcribe` returns the generated `text` (whitespace-stripped, otherwise as decoded), optional `chunks` when timestamps are requested, the task and language, and the model identity. The text is a **generated transcript**: greedy autoregressive decoding with the upstream generation config, no confidence score, no acceptance threshold, and no guarantee against hallucinated words on silence or noise — a deployment that needs an abstain option must build its own detector on its own labelled audio. `language='en'` fixes the decoder's language token; leave it `None` to let the model detect the language. Decoding is deterministic given the same weights, device and library versions; float16 on CUDA and float32 on CPU can differ in near-tied tokens. Look for the transcript next to the reference.

In [ ]:
result = pipe.transcribe(audio_input, language='en', task='transcribe')
print({'task': result['task'], 'language': result['language'], 'device': result['device'], 'source': result['source']})
print('transcript:', result['text'])
print('reference: ', reference)

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. When a reference transcript exists — the public sample's, or `REFERENCE_TEXT` for BYOD — it carries `word_error_rate` (the repository's metric helper: case-folded, punctuation removed, curly apostrophes folded; abbreviations and numbers are **not** normalised, so `Mr.` versus `MISTER` counts as an error) with the verdict `sample-sanity` — one utterance, no dispersion estimate. Without a reference the verdict is `not-measurable` and the report states what would make the task measurable: reference transcripts for several hundred utterances from the deployment domain. The report is written to `outputs/whisper_asr_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, reference, sample_kind=sample_kind)
with open('outputs/whisper_asr_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'not-measurable':
    print('No reference transcript was supplied, so word_error_rate is not computed; the transcript above is sanity evidence only.')

## 8. Export outputs and provenance

Machine-readable JSON preserves the full transcription result, the evaluation report, the input manifest, the sample identity and waveform digest, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, device). The transcript is also written as a plain-text file. No credentials are recorded.

In [ ]:
payload = {
    'prediction': result,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'name': sample_name, 'seconds': round(len(waveform) / sampling_rate, 3), 'sampling_rate': sampling_rate, 'waveform_sha256': sample_sha256, 'reference': reference},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/whisper_asr_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/whisper_asr_transcript.txt', 'w', encoding='utf-8') as handle:
    handle.write(result['text'] + '\n')
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The ASR text is a model-generated transcript with no confidence score; the pipeline ships no threshold and cannot tell a correct word from a fluent hallucination. `word_error_rate`, when shown, is tied to the single demonstrated reference after the stated normalisation and must not be generalised to other languages, speakers, accents, domains, microphones or noise conditions; one utterance is not an error rate. Silence, music, overlapping speakers, code-switching and heavy accents degrade results in ways the pipeline does not detect. The pipeline provides no diarization, speaker identity, biometric inference, streaming or training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated input, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** enable `USE_BYOD` with a recording you have transcribed yourself and paste the transcript into `REFERENCE_TEXT` to see the report switch to `sample-sanity`; set `task='translate'` on non-English audio and compare; pass `return_timestamps=True` to `pipe.transcribe` to get `chunks` with time offsets; record a few seconds of silence and observe what the decoder generates.

## References

- Repository README: https://github.com/kurtvalcorza/whisper-asr-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/whisper-asr-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/whisper-asr-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/openai/whisper-large-v3-turbo
- Upstream code: https://github.com/openai/whisper
- Whisper paper: https://arxiv.org/abs/2212.04356
- Public sample: https://huggingface.co/datasets/hf-internal-testing/librispeech_asr_dummy